In [15]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

In [17]:
# First rows
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [18]:
# Number of rows and columns
print("Shape:", order_reviews.shape)

# Column names
print("Columns:", order_reviews.columns.tolist())

# Data types and non-null counts
order_reviews.info()

# Statistical summary
order_reviews.describe()

Shape: (99224, 7)
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [19]:
date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_columns:
    order_reviews[col] = pd.to_datetime(
        order_reviews[col],
        errors="coerce"
    )

print(order_reviews[date_columns].dtypes)

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [20]:
#counts missing values in each column
order_reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [21]:
print(order_reviews["review_score"].value_counts().sort_index())

print("\nReview score percentages:")
print(
    (order_reviews["review_score"].value_counts(normalize=True)
     .sort_index() * 100).round(2)
)

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Review score percentages:
review_score
1    11.51
2     3.18
3     8.24
4    19.29
5    57.78
Name: proportion, dtype: float64


In [22]:
#check for exact duplicate rows
order_reviews.duplicated().sum()

np.int64(0)

In [23]:
import csv

order_reviews_clean = order_reviews.copy()

order_reviews_clean.to_csv(
    "order_reviews_clean.csv",
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL
)

In [24]:
with open("order_reviews_clean.csv", "r", encoding="utf-8") as f:
    print(f.readline())
    print(f.readline())

"review_id","order_id","review_score","review_comment_title","review_comment_message","review_creation_date","review_answer_timestamp"

"7bc2406110b926393aa56f80a40eba40","73fc7af87114b39712e6da79b0a377eb","4","","","2018-01-18 00:00:00","2018-01-18 21:46:59"



In [25]:
print(order_reviews_clean.shape)
print(order_reviews_clean.columns.tolist())
print(order_reviews_clean["review_id"].head(10))

(99224, 7)
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
0    7bc2406110b926393aa56f80a40eba40
1    80e641a11e56f04c1ad469d5645fdfde
2    228ce5500dc1d8e020d8d1322874b6f0
3    e64fb393e7b32834bb789ff8bb30750e
4    f7c4243c7fe1938f181bec41a392bdeb
5    15197aa66ff4d0650b5434f1b46cda19
6    07f9bee5d1b850860defd761afa7ff16
7    7c6400515c67679fbee952a7525281ef
8    a3f6f7f6f433de0aefbb97da197c554c
9    8670d52e15e00043ae7de4c01cc2fe06
Name: review_id, dtype: str


In [26]:
!pip install sqlalchemy psycopg2-binary

In [40]:
import pandas as pd
from sqlalchemy import create_engine, URL, text
from getpass import getpass

# Enter your PostgreSQL credentials
username = input("PostgreSQL username: atikesanli")
password = getpass("PostgreSQL password: sMCfiJWAsqISPP5w")
host = input("Host [localhost]: data-analytics-course-2.c8g8r1deus2v.eu-central-1.rds.amazonaws.com") or "localhost"
port = input("Port [5432]: ") or "5432"
database = input("Database name: nf_da_onl_en_200726")

# Create connection
url = URL.create(
    "postgresql+psycopg2",
    username=username,
    password=password,
    host=host,
    port=int(port),
    database=database
)

engine = create_engine(url)

# 3. Check the cleaned DataFrame
print("Rows to import:", len(order_reviews_clean))
print("Columns:", order_reviews_clean.columns.tolist())

# 4. Import data and create table automatically
try:
    with engine.begin() as connection:

        order_reviews_clean.to_sql(
            name="olist_order_reviews",
            schema="s_atikesanli",
            con=connection,
            if_exists="fail",
            index=False,
            chunksize=500,
            method="multi"
        )

    print("\nImport completed successfully!")

except Exception as e:
    print("\nImport failed:")
    print(e)

finally:
    engine.dispose()
print("Import completed successfully!")

Rows to import: 98410
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Import completed successfully!
Import completed successfully!


In [30]:
# Check duplicated review IDs
print("Total rows:", len(order_reviews))
print("Unique review IDs:", order_reviews['review_id'].nunique())
print("Duplicated review IDs:", order_reviews['review_id'].duplicated().sum())

# Show all rows with duplicated review IDs
duplicates = order_reviews[
    order_reviews['review_id'].duplicated(keep=False)
]

display(duplicates)

Total rows: 99224
Unique review IDs: 98410
Duplicated review IDs: 814


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
200,28642ce6250b94cc72bc85960aec6c62,e239d280236cdd3c40cb2c033f681d1c,5,NaN,NaN,2018-03-25,2018-03-25 21:03:02
344,a0a641414ff718ca079b3967ef5c2495,169d7e0fd71d624d306f132acd791cbe,5,NaN,NaN,2018-03-04,2018-03-06 20:12:53
346,f4d74b17cd63ee35efa82cd2567de911,f269e83a82f64baa3de97c2ebf3358f6,3,NaN,"A embalagem deixou a desejar, por pouco o prod...",2018-01-12,2018-01-13 18:46:10
360,ecbaf1fce7d2c09bfab46f89065afeaf,2451b9756f310d4cff5c7987b393870d,5,NaN,NaN,2017-07-27,2017-07-28 16:57:18
393,6b1de94de0f4bd84dfc4136818242faa,92acf87839903a94aeca0e5040d99acb,5,NaN,NaN,2018-02-16,2018-02-19 19:04:21
...,...,...,...,...,...,...,...
99108,2c6c08892b83ba4c1be33037c2842294,42ae1967f68c90bb325783ac55d761ce,4,NaN,"Chegou um pouco amassada, mas nada de mais, e ...",2017-07-03,2017-07-05 19:06:59
99124,6ec93e77f444e0b1703740a69122e35d,e1fdc6e9d1ca132377e862593a7c0bd4,5,NaN,Vendedor compromisso do vou o cliente,2017-10-07,2017-10-07 19:47:11
99164,2afe63a67dfd99b3038f568fb47ee761,c5334d330e36d2a810a7a13c72e135ee,5,NaN,"Muito bom, produto conforme anunciado, entrega...",2018-03-03,2018-03-04 22:56:47
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30


In [31]:
# Check exact duplicate rows
print(
    "Exact duplicate rows:",
    order_reviews.duplicated().sum()
)

# Count how many review IDs appear more than once
print(
    "Review IDs appearing more than once:",
    (order_reviews['review_id'].value_counts() > 1).sum()
)

# Display duplicated records
display(
    order_reviews[
        order_reviews['review_id'].duplicated(keep=False)
    ].sort_values('review_id')
)

Exact duplicate rows: 0
Review IDs appearing more than once: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22,2017-07-26 13:41:07


In [32]:
# Select all rows with repeated review IDs
duplicates = order_reviews[
    order_reviews.duplicated(subset='review_id', keep=False)
].copy()

# Count unique values per column for each review ID
differences = duplicates.groupby('review_id').nunique()

# Find review IDs where at least one column differs
different_records = differences[
    (differences > 1).any(axis=1)
]

print("Review IDs with different information:", len(different_records))

display(different_records)

Review IDs with different information: 789


,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
review_id,,,,,,
00130cbe1f9d422698c812ed8ded1919,2,1,0,1,1,1
0115633a9c298b6a98bcbe4eee75345f,2,1,0,0,1,1
0174caf0ee5964646040cd94e15ac95e,2,1,0,1,1,1
017808d29fd1f942d97e50184dfb4c13,2,1,0,0,1,1
0254bd905dc677a6078990aad3331a36,2,1,0,1,1,1
...,...,...,...,...,...,...
fde2e6abaf5bb64f7407a44741c24dec,2,1,0,0,1,1
fde5986d35c89aa1b6ce4149de82a0d3,2,1,0,0,1,1
fe5c833752953fed3209646f1f63b53c,2,1,0,1,1,1


In [33]:
print(order_reviews.columns.tolist())

['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [34]:
# Inspect one duplicated review ID
sample_id = '00130cbe1f9d422698c812ed8ded1919'

display(
    order_reviews[
        order_reviews['review_id'] == sample_id
    ].T
)

,29841,46678
review_id,00130cbe1f9d422698c812ed8ded1919,00130cbe1f9d422698c812ed8ded1919
order_id,04a28263e085d399c97ae49e0b477efa,dfcdfc43867d1c1381bfaf62d6b9c195
review_score,1,1
review_comment_title,NaN,NaN
review_comment_message,"O cartucho ""original HP"" 60XL não é reconhecid...","O cartucho ""original HP"" 60XL não é reconhecid..."
review_creation_date,2018-03-07 00:00:00,2018-03-07 00:00:00
review_answer_timestamp,2018-03-20 18:08:23,2018-03-20 18:08:23


In [35]:
# Check whether each review ID has consistent review information
check_columns = [
    'review_score',
    'review_comment_title',
    'review_comment_message',
    'review_creation_date',
    'review_answer_timestamp'
]

review_consistency = (
    duplicates.groupby('review_id')[check_columns]
    .nunique(dropna=False)
)

print("Review IDs with conflicting review information:",
      (review_consistency > 1).any(axis=1).sum())

Review IDs with conflicting review information: 0


In [36]:
order_reviews_clean = order_reviews.drop_duplicates(
    subset='review_id',
    keep='first'
).copy()

print("Original rows:", len(order_reviews))
print("Cleaned rows:", len(order_reviews_clean))
print("Unique review IDs:", order_reviews_clean['review_id'].nunique())

Original rows: 99224
Cleaned rows: 98410
Unique review IDs: 98410


In [37]:
# Verify cleaned dataset
print("Total rows:", len(order_reviews_clean))

print("Unique review IDs:",
      order_reviews_clean['review_id'].nunique())

print("Duplicated review IDs:",
      order_reviews_clean['review_id'].duplicated().sum())

print("Missing values:")
print(order_reviews_clean.isnull().sum())

Total rows: 98410
Unique review IDs: 98410
Duplicated review IDs: 0
Missing values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       86891
review_comment_message     57742
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [38]:
# Save cleaned dataset as a new CSV
order_reviews_clean.to_csv(
    'order_reviews_clean.csv',
    index=False
)